<p style="text-align: center;"><span style="color: #ff0000;"><strong><span style="font-size: x-large;">
    MODELADO Y VISUALIZACIÓN GRÁFICA</span></strong></span></p>

<p style="text-align: center;"><span style="color: #0000ff;"><strong><span style="font-size: x-large;">PRÁCTICA 4</span></strong></span></p>
<p style="text-align: center;"><span style="color: #0000ff;"><strong><span style="font-size: x-large;">MODELADO CON SUPERFICIES </span></strong></span></p>


Organizamos la práctica según el siguiente índice:<a name="inicio"></a>


<p><span style="color: #0000ff; font-size: large;"><strong>
<a href="#sup">1. Representación gráfica de superficies paramétricas</a></strong> </span></p>

<p><span style="color: #0000ff; font-size: large;"><strong>
<a href="#iso">2. Curvas isoparamétricas</a></strong> </span></p>

<p><span style="color: #0000ff; font-size: large;"><strong>
<a href="#vn">3. Vector normal en un punto</a></strong> </span></p>


<p><span style="color: #0000ff; font-size: large;"><strong>
<a href="#srev">5. Superficies analíticas</a></strong> </span></p>

<p style="margin-left: 20px;">
    <span style="color: #0000ff; font-size: medium;">
        <strong><a href="#srev">5.1. Superficies de revolución</a></strong>
    </span>
</p>

<p style="margin-left: 20px;">
    <span style="color: #0000ff; font-size: medium;">
        <strong><a href="#sr">5.2. Superficies regladas</a></strong>
    </span>
</p>

<p style="margin-left: 20px;">
    <span style="color: #0000ff; font-size: medium;">
        <strong><a href="#sb">5.3. Superficies bilineales</a></strong>
    </span>
</p>

<p><span style="color: #0000ff; font-size: large;"><strong>
<a href="#sbez">4. Superficies de Bézier</a></strong> </span></p>


<p><span style="color: #0000ff; font-size: large;"><strong>
<a href="#tri">6. Triangulación de superficies paramétricas y almacenamiento en ficheros .obj</a></strong> </span></p>

In [ ]:
#Funciones de prácticas anteriores necesarias:

def producto_vectorial(U,V):
    
    # Input: vectores U y V 
    # Output: vector producto vectorial UxV   
    
    W0 = U[1,0]*V[2,0] - U[2,0]*V[1,0]
    W1 = -U[0,0]*V[2,0] + U[2,0]*V[0,0]
    W2 = U[0,0]*V[1,0] - U[1,0]*V[0,0]
    
    pv = matrix([[W0],[W1],[W2],[0]])
    modulo = sqrt((transpose(pv)*pv)[0,0])
    
    return pv/modulo

def dibujar_curva_parametrica(C, a, b, **kwds):

    graf = parametric_plot3d((C[0,0], C[1,0], C[2,0]), (a, b), **kwds)  
    
    return graf

def dibujar_curva_bezier(PC, **kwds):
    
    #Input: Puntos de control de la curva y color
    #Output: gráfica de la curva
    
    C = matrix(PC[:,0:4]*B*T)
    graf = dibujar_curva_parametrica(C, 0, 1, **kwds)
    
    return graf

def dibujar_mallado_poligonal(vertices,caras,**kwds):
    
    # Input: matriz con los vértices del polígono por columnas y matriz de caras.
    # Output: gráfica del mallado.
    
    mallado = 0
    for cara in caras:        
        indices = [n for n in cara if n>-1]
        vertices_de_la_cara = vertices[:,indices]
        mallado = mallado + dibujar_poligono(vertices_de_la_cara,**kwds)
        
    return mallado

def dibujar_poligono(vertices,**kwds):
    
    # Input: matriz con los vértices del polígono por columnas
    # Output: gráfica del polígono con el borde negro de grosor 1
    
    V = list(vertices[0:3].transpose())
    dib = polygon(V,**kwds)
    V.extend(vertices[0:3,0].transpose())
    dib += line3d(V, color ='black', thickness=1)
    return dib

def leer_objeto_obj(path):

    # Input: Ruta donde se encuentra almacenado el archivo .obj
    # Output: Matrices vertices = coordenadas de los vértices por columna y 
    #                  caras = relaciones de los vértices que forman las caras
    
    vertices = []
    caras = []
    
    # Abrir el archivo .obj
    with open(path, 'r') as f:
        for line in f:
            # Leer los vértices
            if line.startswith('v '):
                # Extraemos las coordenadas del vértice (x, y, z)
                v = line.strip().split()[1:]
                # Convertir cada valor de la coordenada flotante a un número racional
                v_racional = [Rational(float(coord)) for coord in v]  # Convertimos a flotante primero y luego a racional
                v_racional.append(1)  # Añadimos la coordenada homogénea (1)
                vertices.append(v_racional)
            
            # Leer las caras
            elif line.startswith('f '):
                # Extraemos los índices de los vértices (restamos 1 porque los índices en .obj empiezan en 1)
                cara = [int(i.split('/')[0]) - 1 for i in line.strip().split()[1:]]
                caras.append(cara)
                
    M = max([len(c) for c in  caras])            
    caras = [c + [-1] * (M - len(c)) for c in caras]
    
    # Convertir las listas a matrices de SageMath
    caras = matrix(len(caras), M,[item for sublist in caras for item in sublist]) 
    vertices = matrix(len(vertices),4,[vertices[i][j] for i in range(len(vertices)) for j in range(4)]).transpose()

    return vertices, caras

In [ ]:
# Estas instrucciones son muy necesarias
u = var('u')
v = var('v')

<a name="sup"></a>
## <span style="color:blue"> 1. Representación gráfica de superficies paramétricas </span>

Dada una superficie paramétrica, $S(u,v)$ para $a\leq u\leq b$ y $c\leq v\leq d$, la siguiente función realiza una representación gráfica: 

In [ ]:
def dibujar_superficie_parametrica(S,a,b,c,d,**kwds):
    
    #Input: S=punto genérico de la superficie, intervalo  [a,b] para el parámetro u, intervalo  [c,d] para el parámetro u
    #Output: gráfica de la curva
        
    graf = parametric_plot3d((S[0,0],S[1,0],S[2,0]), (a,b), (c,d),**kwds)
    
    return graf

### Ejemplo:

Vamos a representar la superficie paramétrica $S(u,v)=\left(\begin{array}{c}
v cos(u)\\
v sen(u)\\
v^2\\
1
\end{array}\right), \, 0 \leq u \leq 2, \, 0 \leq v \leq 2 \pi$

In [ ]:
S = matrix([[v*cos(u)],[v*sin(u)],[v**2],[1]])
show('S(u,v) =', S)

In [ ]:
dib_S = dibujar_superficie_parametrica(S,0,2*pi,0,2, alpha = 0.5) 
dib_S

Volver al [índice](#inicio)

<a name="iso"></a>
## <span style="color:blue"> 2. Curvas isoparamétricas </span>

Dada una superficie paramétrica $S(u,v), \, a\leq u\leq b$ y $c\leq v\leq d$, las curvas isoparamétricas son curvas que se obtienen al fijar uno de los parámetros:

$$C_{u_0}(v) = S(u = u_0,v), \quad c\leq v\leq d$$
$$C_{v_0}(u) = S(u,v = v_0), \quad a\leq u\leq b$$

### Ejemplo:

Dada la superficie paramétrica del ejemplo anterior, vamos a calcular y a representar las curvas isoparamétricas que se obtienen al fijar el parámetro $u = \pi$ y $v = 1$.

In [ ]:
show('S(u,v) =', S)

In [ ]:
u0 = pi
v0 = 1

#Curvas isoparamétricas:
Cu0 = S(u = u0, v = v)
Cv0 = S(u = u, v = v0)

show('Curva Cu0(v) = ', Cu0)
show('Curva Cv0(u) = ', Cv0)

In [ ]:
dib_C1 = dibujar_curva_parametrica(Cu0, 0, 2, color = 'red', thickness = 3)
dib_C2 = dibujar_curva_parametrica(Cv0, 0, 2*pi, color = 'green', thickness = 3)
dib_S + dib_C1 + dib_C2

La siguiente función dibuja tantas curvas isoparamétricas de una superficie como se le indique en los argumentos de entrada n_u0 y n_v0:

In [ ]:
def dibujar_curvas_isoparametricas(S,a,b,n_u0,c,d,n_v0):

    #Input: S = superficie, intervalo [a,b] para el parámetro u, número de curvas n_u0 
    #       intervalo [c,d] para el parámetro v, número de curvas n_v0 
    #Output: gráfica de las curvas isoparamétricas
    
    graf_curvas_iso = Graphics()
    
    paso_u0 = (b-a)/(n_u0)
    paso_v0 = (d-c)/(n_v0)

    for u0 in srange(a, b + paso_u0, paso_u0): 
        C = S(u = u0,v = v)
        graf_curvas_iso += dibujar_curva_parametrica(C, c, d, color = 'green')
    
    for v0 in srange(c, d + paso_v0, paso_v0):
        C = S(u = u,v = v0)
        graf_curvas_iso += dibujar_curva_parametrica(C, a, b, color = 'red')
    
    return graf_curvas_iso

### Ejemplo:

Vamos a representar 20 curvas isoparamétricas para cada uno de los parámetros:

In [ ]:
dib = dibujar_curvas_isoparametricas(S,0,2*pi,20,0,2, 20)
dib

Volver al [índice](#inicio)

<a name="vn"></a>
## <span style="color:blue"> 3. Vector normal en un punto </span>

Vamos a calcular paso a paso el vector normal a la superficie de los ejemplos anteriores en el punto $S(1,\pi)$: 

In [ ]:
u0 = pi
v0 = 1

P = S(u = u0,v = v0)
show('Punto S(pi,1) = ',P)

In [ ]:
#1. Se calculan las curvas isoparamétricas
Cu0 = S(u = u0,v = v)
Cv0 = S(u = u,v = v0)

show('Curva Cu0 = ', Cu0)
show('Curva Cv0 = ', Cv0)

In [ ]:
#2. Derivadas de las curvas isoparamétricas:
Cu0_der = diff(Cu0,v)
Cv0_der = diff(Cv0,u)

show('Derivada de Cu0 = ', Cu0_der)
show('Derivada de Cv0 =', Cv0_der)

In [ ]:
#3. Vectores tangentes a las curvas en el punto P:
vector_tg_u0 = Cu0_der(v = v0)
vector_tg_v0 = Cv0_der(u = u0)

show('Vector tangente a Cu0(v) = ', vector_tg_u0)
show('Vector tangente a Cv0(u) = ', vector_tg_v0)

In [ ]:
#4. Vector normal unitario:
N = producto_vectorial(vector_tg_u0,vector_tg_v0)
show('Vector normal = ', N) 

Representamos gráficamente el vector normal junto con las curvas isoparamétricas ese vector podría apuntar "hacia fuera" o "hacia dentro" de la superficie, eso sólo depende del orden que hayamos dado a los vectores tangentes en el producto vectorial.

In [ ]:
#Para representarlo calculamos el otro punto extremo del vector como Q = P + N (ya que N = PQ)
Q = P + N
dib_N = arrow3d(P.column(0)[0:3], Q.column(0)[0:3], width = 2) 

dib_S + dib_C1 + dib_C2 + dib_N

Vemos que el vector normal N apunta "hacia dentro". Para encontrar un vector normal a la superficie que apunte "hacia fuera" solo hay que intercambiar los factores del producto vectorial: 

In [ ]:
Q = P + N2
dib_N2 = arrow3d(P.column(0)[0:3], Q.column(0)[0:3], width = 2)

dib_S + dib_C1 + dib_C2 + dib_N2

Volver al [índice](#inicio)

<a name="srev"></a>
## <span style="color:blue"> 5. Superficies analíticas: ejemplos de algunas superficies obtenidas usando transformaciones geométricas sobre curvas. </span>

<a name="srev"></a>
## <span style="color: #0000ff; font-size: large;">5.1. Superficies de revolución</span>

Las superficies de revolución se obtienen al rotar una curva $C(u)$, $a \leq u \leq b$, sobre un eje coplanario con dicha curva. Sin pérdida de generalidad, asumimos en lo que sigue que se trata del eje $z$. Por tanto, la superficie viene dada por: 

$$S(u,v) = R(v)C(u), \quad a \leq u \leq b, \, 0 \leq v \leq 2\pi$$

siendo $R(v) = \left(\begin{array}{cccc}
cos(v) & -sen(v) & 0 & 0\\
sen(v) & cos(v) & 0 & 0\\
0 & 0 & 1 & 0\\
0 & 0 & 0 & 1\\
\end{array}\right)$


### Ejemplo:

Sea $C(u)$,  $0 \leq u \leq 1$, la curva cúbica de Bézier con puntos de control $PC = \{P_0, P_1, P_2, P_3\}$ dados por:

$$P_0=\left(\begin{array}{c}3\\0\\0\\1\end{array}\right)\;
P_1=\left(\begin{array}{c}2\\0\\1\\1\end{array}\right)\;
P_2=\left(\begin{array}{c}2\\0\\2\\1\end{array}\right)\;
P_3=\left(\begin{array}{c}3\\0\\4\\1\end{array}\right)
$$

a) Calculamos la ecuación paramétrica de la superficie $S(u,v)$, $0 \leq u \leq 1$ y $0 \leq v \leq 2\pi$, que se obtiene al rotar la curva $C(u)$ alrededor del eje $z$.

b) Representamos gráficamente la curva $C(u)$, así como de la superficie de revolución obtenida.

In [ ]:
u = var('u')
v = var('v')

#Definimos la matriz con los puntos de control, la matriz base de Bézier B y la base canónica U
PC = PC = matrix([[1,0,4,1],[0,0,3,1],[3,0,1,1],[1,0,0,1]]).transpose()
B = matrix([[-1,3,-3,1],[3,-6,3,0],[-3,3,0,0],[1,0,0,0]])    
U = matrix([[u^3],[u^2],[u],[1]])
#-------------------------------------------------------------------------------------
#Definimos la curva como resultado de la multiplicación PC*B*U 
C = matrix(PC*B*U)
show('Curva base C(u) =', C)
#-------------------------------------------------------------------------------------
#Definimos la superficie de revolución 
R = matrix([[cos(v),-sin(v),0,0],[sin(v),cos(v),0,0],[0,0,1,0],[0,0,0,1]])
S = R*C
show('S(u,v) =', S)

In [ ]:
dib_C = dibujar_curva_parametrica(C,0,1,color = 'red', thickness = 3) 
dib_S = dibujar_superficie_parametrica(S,0,1,0,2*pi, alpha = 0.8) 
dib_C + dib_S

Volver al [índice](#inicio)

<a name="sr"></a>
## <span style="color: #0000ff; font-size: large;">5.2. Superficies regladas </span>

Las superficies regladas están definidas por la interpolación lineal de dos curvas $C_1(u)$ y $C_2(u)$, para $a \leq u \leq b$ (ambas curvas en el mismo rango de variación del parámetro). Por tanto, la superficie viene dada por:

$$S(u,v) = (1-v) C_1(u) + v C_2(u), \quad a \leq u \leq b, \, 0 \leq v \leq 1$$

### Ejemplo:

Se consideran dos circunferencias centradas en los puntos
$P_1=\left(\begin{array}{c}0\\0\\0\\1\end{array}\right)$ y $P_2=\left(\begin{array}{c}0\\0\\2\\1\end{array}\right)$
con
radios $r_1 = 3$ y $r_2 = 1$ y situadas en los planos $z = 0$ y $z = 2$ respectivamente. Vamos a obtener las ecuaciones
paramétricas de la superficie reglada que las interpola y su representación gráfica.

In [ ]:
C1 = matrix([[3*cos(u)],[3*sin(u)],[0],[1]])
C2 = matrix([[cos(u)],[sin(u)],[2],[1]])
S = (1-v)*C1 + v*C2
show('S(u,v) = ', expand(S))

In [ ]:
dib_C1 = dibujar_curva_parametrica(C1, 0, 2*pi, color = 'red', thickness = 3)
dib_C2 = dibujar_curva_parametrica(C2, 0, 2*pi, color = 'green', thickness = 3) 
dib_S = dibujar_superficie_parametrica(S, 0, 2*pi, 0, 1, alpha = 0.6)
dib_S + dib_C1 + dib_C2 

Volver al [índice](#inicio)

<a name="sb"></a>
## <span style="color: #0000ff; font-size: large;">5.3. Superficies bilineales </span>

Un caso destacable son las **superficies bilineales**: superficies regladas interpolando 4 puntos $P_0, P_1, P_2, P_3$. En dicho caso, la superficie se puede expresar de forma matricial:

$$S(u,v) = (1-v \quad v) \left(\begin{array}{cc}P_0 & P_1\\P_2 & P_3\end{array}\right) \left(\begin{array}{c}1-u\\u\end{array}\right), \quad 0 \leq u \leq 1, \, 0 \leq v \leq 1$$

### Ejemplo:

Sea una superficie bilineal con puntos de control $PC = \{P_0, P_1, P_2, P_3\}$ dados por:

$$P_0=\left(\begin{array}{c}0\\0\\0\\1\end{array}\right)\;
P_1=\left(\begin{array}{c}6\\0\\6\\1\end{array}\right)\;
P_2=\left(\begin{array}{c}2\\4\\4\\1\end{array}\right)\;
P_3=\left(\begin{array}{c}5\\4\\4\\1\end{array}\right)
$$

Vamos a calcular la ecuación paramétrica de la superficie $S(u,v)$, $0 \leq u, v \leq 1$. 

In [ ]:
#Procedemos de la misma forma que para las superficies de Bézier
P0 = matrix([0,0,0,1]).transpose()
P1 = matrix([6,0,6,1]).transpose()
P2 = matrix([2,4,4,1]).transpose()
P3 = matrix([5,4,4,1]).transpose()

PCx = matrix([[P0[0,0],P1[0,0]],[P2[0,0],P3[0,0]]])
PCy = matrix([[P0[1,0],P1[1,0]],[P2[1,0],P3[1,0]]])
PCz = matrix([[P0[2,0],P1[2,0]],[P2[2,0],P3[2,0]]])

show('PCx = ', PCx, ' ; PCy = ', PCy, ' ; PCz = ', PCz)

U = matrix([[1-u],[u]])
V = matrix([[1-v,v]])

S = ((V*PCx*U).augment(V*PCy*U).augment(V*PCz*U).augment(matrix([[1]]))).transpose()
show('S(u,v) = ', S)

In [ ]:
dibujar_superficie_parametrica(S,0,1,0,1, alpha = 0.8) 

## <span style="color:red">Ejercicio:</span>

Con los datos del ejemplo anterior:
    
- Calcula los dos vectores tangentes y el vector normal unitario $N$ a la superficie en el punto $S(\frac{2}{3}, \frac{1}{2})$.

- Comprueba que el vector normal unitario a la superficie $F(u,v)=S(v,u)$ en el punto $F(\frac{1}{2}, \frac{2}{3})$ vale $-N$.

Volver al [índice](#inicio)

<a name="sbez"></a>
## <span style="color:blue"> 4. Superficies de Bézier </span>


**Superficies bi-cúbicas de Bézier:**

$$S(u,v) = \left(\begin{array}{cccc}u^3 & u^2 & u & 1\end{array}\right) B \left(\begin{array}{cccc}P_{00} & P_{10} & P_{20} & P_{30}\\P_{01} & P_{11} & P_{21} & P_{31}\\P_{02} & P_{12} & P_{22} & P_{32}\\P_{03} & P_{13} & P_{23} & P_{33}\end{array}\right) B \left(\begin{array}{c}v^3\\v^2\\v\\1\end{array}\right), \quad 0 \leq u \leq 1, \, 0 \leq v \leq 1$$

### Ejemplo:

Consideramos la superficie de Bezier $S(u,v)$, $0 \leq u,v \leq 1$, con puntos de control: 

$$PC = \left(P_{00} \; P_{01} \; P_{02} \; P_{03} \; P_{10} \; P_{11} \; P_{12} \; P_{13} \; P_{20} \; P_{21} \; P_{22} \; P_{23} \; P_{30} \; P_{31} \; P_{32} \; P_{33}\right)$$ 

$$PC=\left(\begin{array}{cccccccccccccccc}
-15 & -15 & -15 & -15 & -5 & -5 & -5 & -5 & 5 & 5 & 5 & 5 & 15 & 15 & 15 & 15\\
0 & 5 & 5 & 0 & 5 & 5 & 5 & 5 & 5 & 5 & 5 & 5 & 0 & 5 & 5 & 0\\
15 & 5 & -5 & -15 & 15 & 5 & -5 & -15 & 15 & 5 & -5 & -15 & 15 & 5 & -5 & -15\\
1  &1  &1  &1  &1  &1  &1  &1  &1  &1  &1  &1  &1  &1  &1  &1  
\end{array}\right)$$

Vamos a determinar el punto $Q$ de la superficie que se obtiene para los valores de los parámetros $u = v = \frac{1}{2}$.

In [ ]:
PC = matrix([[-15, -15 , -15 , -15, -5 , -5 ,-5 , -5, 5 , 5, 5 , 5 ,15 , 15, 15, 15],
            [0 ,5 , 5 , 0 , 5 , 5, 5 , 5 , 5 , 5 , 5 , 5 , 0, 5, 5 , 0],
            [15 , 5 , -5 ,-15 , 15 , 5 , -5 , -15 , 15 ,5 , -5 , -15 , 15 , 5 , -5 , -15],
            [1  ,1  ,1  ,1  ,1  ,1 ,1 ,1  ,1  ,1  ,1  ,1 ,1  ,1 ,1  ,1 ]])

Definimos la matriz $\left(\begin{array}{cccc}P_{00} & P_{10} & P_{20} & P_{30}\\P_{01} & P_{11} & P_{21} & P_{31}\\P_{02} & P_{12} & P_{22} & P_{32}\\P_{03} & P_{13} & P_{23} & P_{33}\end{array}\right)$. Para ello, definimos 3 matrices bidimensionales, $PCx, PCy, PCz$, correspondientes a las coordenadas $x, y, z$ de los puntos. Por ejemplo, la matriz $PCx$ vendría dada por:

$$PCx = \left(\begin{array}{cccc}P_{00x} & P_{10x} & P_{20x} & P_{30x}\\P_{01x} & P_{11x} & P_{21x} & P_{31x}\\P_{02x} & P_{12x} & P_{22x} & P_{32x}\\P_{03x} & P_{13x} & P_{23x} & P_{33x}\end{array}\right) = \left(\begin{array}{cccc}-15 & -5 & 5 & 15\\-15 & -5 & 5 & 15\\-15 & -5 & 5 & 15\\-15 & -5 & 5 & 15\end{array}\right)$$

In [ ]:
row = 0
PCx = matrix([[PC[row,0],PC[row,4],PC[row,8],PC[row,12]],
                  [PC[row,1],PC[row,5],PC[row,9],PC[row,13]],
                  [PC[row,2],PC[row,6],PC[row,10],PC[row,14]],
                  [PC[row,3],PC[row,7],PC[row,11],PC[row,15]]])
row = 1
PCy = matrix([[PC[row,0],PC[row,4],PC[row,8],PC[row,12]],
                  [PC[row,1],PC[row,5],PC[row,9],PC[row,13]],
                  [PC[row,2],PC[row,6],PC[row,10],PC[row,14]],
                  [PC[row,3],PC[row,7],PC[row,11],PC[row,15]]])
row = 2
PCz = matrix([[PC[row,0],PC[row,4],PC[row,8],PC[row,12]],
                  [PC[row,1],PC[row,5],PC[row,9],PC[row,13]],
                  [PC[row,2],PC[row,6],PC[row,10],PC[row,14]],
                  [PC[row,3],PC[row,7],PC[row,11],PC[row,15]]])

show('PCx = ', PCx, '; PCy = ', PCy, ' ; PCz = ', PCz)

Definimos $U = \left(\begin{array}{cccc}u^3 & u^2 & u & 1\end{array}\right)$ y $V = \left(\begin{array}{c}v^3\\v^2\\v\\1\end{array}\right)$

In [ ]:
U = matrix([[u^3, u^2, u, 1]])
V = matrix([[v^3],[v^2],[v],[1]])

Por último, definimos la ecuación paramétrica: $S(u,v) = \left(\begin{array}{c}
U \cdot B \cdot PCx \cdot B \cdot V\\
U \cdot B \cdot PCy \cdot B \cdot V\\
U \cdot B \cdot PCz \cdot B \cdot V\\
1\end{array}\right)$

In [ ]:
B = matrix([[-1,3,-3,1],[3,-6,3,0],[-3,3,0,0],[1,0,0,0]])  

Sx = U*B*PCx*B*V
Sy = U*B*PCy*B*V
Sz = U*B*PCz*B*V

S = matrix([[Sx[0,0]],[Sy[0,0]],[Sz[0,0]],[1]])

show('S(u,v) = ',S)

In [ ]:
Q = S(u = 1/2, v = 1/2)
show('Q = ', Q)

In [ ]:
dib_S = dibujar_superficie_parametrica(S,0,1,0,1, alpha = 0.8) 
dib_S + point(Q.column(0)[0:3], size = 200, color = 'cyan')

Definimos ahora una función para representar gráficamente la malla de puntos de control:

In [ ]:
def dibujar_malla_PC(PC,**kwds):
    
    P0 = [n(PC.column(i)[0:3]) for i in range(4)]
    P1 = [n(PC.column(i)[0:3]) for i in range(4,8)]
    P2 = [n(PC.column(i)[0:3]) for i in range(8,12)]
    P3 = [n(PC.column(i)[0:3]) for i in range(12,16)]

    P4 = [n(PC.column(i)[0:3]) for i in [0,4,8,12]]
    P5 = [n(PC.column(i)[0:3]) for i in [1,5,9,13]]
    P6 = [n(PC.column(i)[0:3]) for i in [2,6,10,14]]
    P7 = [n(PC.column(i)[0:3]) for i in [3,7,11,15]]
    
    P = []
    P.extend([P0,P1,P2,P3,P4,P5,P6,P7])
    
    G = Graphics()
    for i in range(8):
        G += points(P[i],**kwds) + line3d(P[i],**kwds)
    
    return G

### Ejemplo:

Vamos a representar la superficie del ejemplo anterior junto con la malla de puntos de control. 

In [ ]:
dib_malla = dibujar_malla_PC(PC, size = 100, color = 'red', thickness = 1)
dib_S + dib_malla

Modificamos el punto de control $P_{02}$:

In [ ]:
PC[:,2] = vector([-20,15,-5,1])

row = 0
PCx = matrix([[PC[row,0],PC[row,4],PC[row,8],PC[row,12]],
                  [PC[row,1],PC[row,5],PC[row,9],PC[row,13]],
                  [PC[row,2],PC[row,6],PC[row,10],PC[row,14]],
                  [PC[row,3],PC[row,7],PC[row,11],PC[row,15]]])
row = 1
PCy = matrix([[PC[row,0],PC[row,4],PC[row,8],PC[row,12]],
                  [PC[row,1],PC[row,5],PC[row,9],PC[row,13]],
                  [PC[row,2],PC[row,6],PC[row,10],PC[row,14]],
                  [PC[row,3],PC[row,7],PC[row,11],PC[row,15]]])
row = 2
PCz = matrix([[PC[row,0],PC[row,4],PC[row,8],PC[row,12]],
                  [PC[row,1],PC[row,5],PC[row,9],PC[row,13]],
                  [PC[row,2],PC[row,6],PC[row,10],PC[row,14]],
                  [PC[row,3],PC[row,7],PC[row,11],PC[row,15]]])

Sx = U*B*PCx*B*V
Sy = U*B*PCy*B*V
Sz = U*B*PCz*B*V

S_mod = matrix([[Sx[0,0]],[Sy[0,0]],[Sz[0,0]],[1]])

dibujar_superficie_parametrica(S_mod,0,1,0,1, color = 'green', alpha = 0.5) + dibujar_malla_PC(PC, size = 100, color = 'green', thickness = 1) + dib_S


## <span style="color:red">Ejercicio:</span>

Considera la superficie de Bézier y el punto $Q$ del ejemplo anterior. 

- Calcular y representar gráficamente las curvas isoparamétricas que pasan por $Q$. 
- Calcular los vectores tangentes y el vector normal unitario a la superficie en $Q$.

Volver al [índice](#inicio)

<a name="tri"></a>
## <span style="color:blue"> 6. Triangulación de superficies paramétricas y almacenamiento en ficheros .obj </span>

*Nota: Este apartado es opcional en el sentido de que no habrña ninguna pregunta en el cuestionario sobre él.*


Dada una superficie $S(u,v)$, el mallado triangular que modela dicha superficie se obtiene a partir de las curvas isoparamétricas como sigue:

- Los vértices del mallado se corresponderán con las intersecciones de dichas curvas.
- Los triángulos del mallado se corresponderán con los resultantes de dividir los cuadrilateros que se forman con las intersecciones de las curvas.

Junto con los vértices y caras del mallado, se almacena también el vector normal a los vértices.

### Ejemplo:

Consideremos la siguiente superficie paramétrica:

$$S(u,v)=\left(\begin{array}{cccc}\cos(v)&-\sin(v)&0&0\\ \sin(v)&\cos(v)&0&0\\0&0&1&0\\0&0&0&1\end{array}\right)\left(\begin{array}{c}\sqrt{u^2+1}\\0\\u\\1\end{array}\right)$$
siendo $-1\leq u\leq 1$ y $0\leq v\leq 2\pi$.

In [ ]:
S = matrix([[cos(v),-sin(v),0,0],[sin(v),cos(v),0,0],[0,0,1,0],[0,0,0,1]])*matrix([[sqrt(u^2+1)],[0],[u],[1]])
dib = dibujar_superficie_parametrica(S,-1,1,0,2*pi, alpha = 0.8) 
dib

In [ ]:
def mallado_superficie(S,a,b,c,d,n_u0,n_v0):
    
    paso_u0 = (b-a)/n_u0
    paso_v0 = (d-c)/n_v0

    #Inicializamos la matriz de vertices y vector normal
    vertices = matrix(RR,4,0)
    normales = matrix(RR,4,0)

    for u0 in srange(a, b+paso_u0, paso_u0): 
        for v0 in srange(c, d+paso_v0, paso_v0): 
            #Vértice
            V = S(u = u0,v = v0)
            #almacenamos el vértice en la matriz de vértices
            vertices = vertices.augment(V.n())        
            #Calculamos el vector normal a la superficie en el vértice v
            #Curvas isoparamétricas:
            C_v0 = S(u = u,v = v0)
            C_u0 = S(u = u0,v = v)
            #Derivadas de las curvas isoparamétricas:
            C_v0_der = diff(C_v0,u)
            C_u0_der = diff(C_u0,v)
            #Vectores tangentes a las curvas en el punto P:
            Vector_v0 = C_v0_der(u = u0)
            Vector_u0 = C_u0_der(v = v0)
            #vector normal unitario
            N = producto_vectorial(Vector_u0,Vector_v0)
            modulo = sqrt((transpose(N)*N)[0,0])
            N = N/modulo
            #almacenamos el vector normal en la matriz correspondiente
            normales = normales.augment(N.n())
            
        caras = matrix(ZZ,0,3)

    #inicializamos los índices para el cálculo de las caras
    i = 0
    for i in range(n_v0): 
        j = 0
        for j in range(n_u0):         
            #construimos las 2 caras que forman el cuadrilátero con esquina en el vértice V        
            cara1 = matrix([[i*(n_v0+1)+j,i*(n_u0+1)+j+1,(i+1)*(n_v0+1)+j]])
            cara2 = matrix([[(i+1)*(n_v0+1)+j,i*(n_u0+1)+j+1,(i+1)*(n_v0+1)+j+1]])
            caras = caras.stack(cara1).stack(cara2) 
            j = j+1
        i = i+1
            
    return vertices,normales,caras

In [ ]:
vertices,normales,caras = mallado_superficie(S,-1,1,0,2*pi,5,5)

In [ ]:
dibujar_mallado_poligonal(vertices,caras)

In [ ]:
def almacenar_objeto_obj(vertices,normales,caras,mi_archivo):

    # Input: Matrices vertices = coordenadas de los vértices por columna y 
    #                 vnormales = coordenadas de los vectores normales a la superficie por columna y 
    #                 caras = relaciones de los vértices que forman las caras
    # Output: mi_archivo.obj   

    # Abre el archivo en modo escritura, creándolo si no existe
    with open('mi_archivo.obj', 'w') as f:
        pass #vacio el archivo si existe
        # Escribir los vértices
        for i in range(vertices.ncols()):
            # Extraemos las tres primeras coordenadas del vértice (se ignora la coordenada homogénea)
            x = vertices[0, i].n()
            y = vertices[1, i].n()
            z = vertices[2, i].n()
            f.write("v {} {} {}\n".format(x, y, z))
    
        # Escribir vector normal
        for i in range(normales.ncols()):
            # Extraemos las tres primeras coordenadas del vértice (se ignora la coordenada homogénea)
            x = normales[0, i].n()
            y = normales[1, i].n()
            z = normales[2, i].n()
            f.write("vn {} {} {}\n".format(x, y, z))
    
    
        # Escribir las caras
        for row in caras.rows():
            # Filtrar los índices válidos (diferentes de -1) y sumarle 1, pues en .obj los índices empiezan en 1
            face_indices = [str(int(idx) + 1) for idx in row if idx != -1]
            if face_indices:
                f.write("f " + " ".join(face_indices) + "\n")


In [ ]:
almacenar_objeto_obj(vertices,normales,caras,'mi_archivo')

In [ ]:
vertices,caras = leer_objeto_obj('mi_archivo.obj')

In [ ]:
dibujar_mallado_poligonal(vertices,caras)
dib

## <span style="color:red">Ejercicio:</span>

Sigue los mismos pasos y almacena una triangulación de la superficie del ejemplo, pero con más resolución (más cantidad de vértices y aristas).

Volver al [índice](#inicio)